# Step 7: Loan Data Wrangling and Exploratory Analysis

This notebook inspects and cleans a working copy of the real `data/raw/loan_data.csv` dataset. It preserves the raw file and does not train a loan model. It reports observed values and derives all statistics from the loaded data.

The dataset publisher's schema identifies `loan_status` as the target and defines 0 as non-default and 1 as default. The code below verifies the column and observed encodings before using that interpretation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from src.data_loader import RAW_DATA_DIR, load_loan_dataset
from src.loan_features import (
    LOAN_SOURCE_URL,
    LOAN_TARGET_COLUMN,
    clean_loan_data,
    summarize_loan_data,
)

source_path = RAW_DATA_DIR / "loan_data.csv"
df = load_loan_dataset()
raw_copy = df.copy(deep=True)
summary = summarize_loan_data(df)

print(f"Source: {source_path}")
print(f"Dataset source: {LOAN_SOURCE_URL}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Observed target: {summary['target_column']}")
print(f"Verified target values: {sorted(pd.to_numeric(df[LOAN_TARGET_COLUMN]).dropna().unique().tolist())}")
print("Target meaning:", summary["target_meanings"])
display(df.head())

## 1. Schema, missingness, duplicates, and categories

The duplicate count is for exact full-row matches only. This file has no applicant ID, so it cannot resolve same-applicant records that differ in content. Category values are printed from the source rather than presumed.

In [ ]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_nonmissing": df.nunique(dropna=True),
})
display(schema)
print(f"Exact duplicate rows: {int(df.duplicated().sum()):,}")
for column in df.select_dtypes(include=["object", "str"]):
    print(f"\n{column} values:")
    display(df[column].value_counts(dropna=False).rename("rows").to_frame())

display(summary["target_distribution"])

## 2. Non-destructive cleaning and data-quality flags

Numeric missing values are imputed with medians and categorical missing values with the most frequent category in this EDA-only fit. Categories are trimmed and case-normalized. Exact duplicate source rows are removed before imputation so distinct records that happen to become equal after filling missing values are retained. The target is preserved. IQR extremes and credit-consistency checks are flags; by default they do not cap or delete financial values.

In [ ]:
quality = clean_loan_data(df)
cleaned = quality.cleaning.data

print("Rows before / after:", quality.cleaning.before_report["rows"], quality.cleaning.after_report["rows"])
print("Missing cells before / after:", quality.cleaning.before_report["missing_cells"], quality.cleaning.after_report["missing_cells"])
print("Exact duplicate rows removed:", quality.cleaning.before_report["duplicate_rows"])
print("Target retained:", LOAN_TARGET_COLUMN in cleaned.columns)
print("Target missing values remain:", int(cleaned[LOAN_TARGET_COLUMN].isna().sum()))
print("IQR outliers flagged by feature:")
display(quality.cleaning.outlier_flags.sum().loc[lambda values: values.gt(0)].astype(int).rename("flagged_rows").to_frame())
print("Credit consistency review flags:")
display(quality.consistency_flags.sum().loc[lambda values: values.gt(0)].astype(int).rename("flagged_rows").to_frame())
print("Category validation findings:")
display(quality.cleaning.corrupted_categories)
display(cleaned.head())

## 3. Default and non-default distribution

The source documentation defines the target encoding. Counts and percentages below are computed from the loaded data, not illustrative values.

In [ ]:
target_distribution = summary["target_distribution"].copy()
target_distribution["outcome"] = target_distribution["loan_status"].map({0: "Non-default (0)", 1: "Default (1)"})
display(target_distribution)
px.bar(target_distribution, x="outcome", y="count", color="outcome", text="count", title="Observed loan_status distribution").show()

## 4. Income and loan amount distributions

Both monetary fields are right-skewed; the plots use logarithmic x-axes for readability while summaries remain in original units.

In [ ]:
outcome_data = df.assign(outcome=df[LOAN_TARGET_COLUMN].map({0: "Non-default", 1: "Default"}))
print("Income and loan amount summary by target:")
display(summary["median_financial_values_by_status"])
display(df[["person_income", "loan_amnt"]].describe(percentiles=[.01, .25, .5, .75, .95, .99]).T)
px.histogram(outcome_data, x="person_income", color="outcome", barmode="overlay", opacity=.65, nbins=60, log_x=True, title="Annual income distribution").show()
px.histogram(outcome_data, x="loan_amnt", color="outcome", barmode="overlay", opacity=.65, nbins=60, log_x=True, title="Loan amount distribution").show()

## 5. Employment information

Employment length is reported in years. The table shows observed default rates across descriptive employment-length bands and by home-ownership category; these are associations, not causal effects.

In [ ]:
display(summary["employment_default_rates"])
display(summary["default_rates_by_category"]["person_home_ownership"])
px.histogram(outcome_data, x="person_emp_length", color="outcome", barmode="overlay", opacity=.65, nbins=40, title="Employment length (years)").show()
px.bar(summary["employment_default_rates"], x="employment_years", y="default_rate", text=summary["employment_default_rates"]["default_rate"].map(lambda value: f"{value:.1%}"), title="Default rate by employment-length band").show()

## 6. Credit-history distributions

The source describes `cb_person_default_on_file` as historical default; its observed values are checked before being interpreted as Y/N. Credit-history length is also examined as a numeric duration.

In [ ]:
display(summary["default_rates_by_category"]["cb_person_default_on_file"])
px.histogram(outcome_data, x="cb_person_cred_hist_length", color="outcome", barmode="overlay", opacity=.65, nbins=30, title="Credit history length (years)").show()
px.bar(summary["default_rates_by_category"]["cb_person_default_on_file"], x="cb_person_default_on_file", y="default_rate", text=summary["default_rates_by_category"]["cb_person_default_on_file"]["default_rate"].map(lambda value: f"{value:.1%}"), title="Default rate by prior default recorded").show()

## 7. Income and loan amount relationship

Pearson and Spearman correlations are computed from observed income/loan-amount pairs. A sample is used only for the scatter plot to keep notebook rendering responsive; correlation values use every complete pair.

In [ ]:
print("Income / loan amount correlation:", summary["income_loan_correlation"])
scatter_data = outcome_data.sample(n=min(5000, len(outcome_data)), random_state=42)
px.scatter(scatter_data, x="person_income", y="loan_amnt", color="outcome", opacity=.55, log_x=True, log_y=True, hover_data=["loan_percent_income", "loan_grade", "loan_intent"], title="Income versus loan amount (5,000-row maximum visualization sample)").show()

## 8. Numeric correlations

The heatmap uses numeric source fields, including `loan_status`. Correlations describe pairwise linear association and are not causal evidence.

In [ ]:
numeric_correlations = df.select_dtypes(include="number").corr(method="pearson")
px.imshow(numeric_correlations, aspect="auto", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Pearson correlation matrix").show()
display(summary["numeric_correlations_with_target"].rename("pearson_correlation").to_frame())

## 9. Outliers and inconsistent credit data

IQR bounds are descriptive, data-derived review thresholds. Extreme income and loan values are not automatically invalid and remain in the cleaned working copy by default. Separate domain checks flag impossible-looking age/employment combinations and differences between the recorded loan-to-income field and the ratio computed from the two source values; these checks do not guess which conflicting source field is correct.

In [ ]:
outlier_features = [column for column in ("person_income", "loan_amnt", "loan_int_rate") if column in df.columns]
box_data = outcome_data.melt(id_vars="outcome", value_vars=outlier_features, var_name="feature", value_name="value")
px.box(box_data, x="feature", y="value", color="outcome", points="outliers", log_y=True, title="Income, loan, and interest-rate outliers (log y-axis)").show()
px.box(outcome_data, x="outcome", y="person_emp_length", points="outliers", title="Employment-length outliers (linear axis; includes zero years)").show()
flagged_consistency = quality.consistency_flags.sum().astype(int).sort_values(ascending=False)
display(flagged_consistency.loc[flagged_consistency.gt(0)].rename("rows flagged").to_frame())
review_rows = quality.consistency_flags.any(axis=1)
display(pd.concat([df.loc[review_rows], quality.consistency_flags.loc[review_rows]], axis=1).head(20))

## 10. Findings generated from the loaded dataset

These findings are formatted from the actual input and analysis outputs. Outlier/consistency counts are flags for review; do not infer invalidity or causation solely from them.

In [ ]:
print(f"Rows: {len(df):,}; columns: {len(df.columns)}")
print(f"Target `{LOAN_TARGET_COLUMN}` counts: {dict(zip(summary['target_distribution']['meaning'], summary['target_distribution']['count']))}")
print(f"Missing cells: {int(df.isna().sum().sum()):,}; exact duplicate rows: {int(df.duplicated().sum()):,}")
print(f"Median income / loan amount by target:\n{summary['median_financial_values_by_status']}")
print(f"Income-loan Pearson r={summary['income_loan_correlation']['pearson']:.3f}; Spearman rho={summary['income_loan_correlation']['spearman']:.3f}")
print(f"The largest absolute numeric correlation with `{LOAN_TARGET_COLUMN}` is `{summary['numeric_correlations_with_target'].index[0]}` ({summary['numeric_correlations_with_target'].iloc[0]:.3f}).")
print("Cleaning log:")
for entry in quality.cleaning.log:
    print("-", entry)
assert df.equals(raw_copy), "The raw input frame was modified"
print("Raw input frame unchanged; no loan models trained.")